In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START , END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.prompts import PromptTemplate
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

class Tweetstate(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]
    draft : str
    final : str

In [10]:
def generate_draft(state: Tweetstate):
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],
        "draft": response.content
    }
def human_review(state: Tweetstate):
    print("\n--- HUMAN REVIEW ---")
    print("Draft answer:\n")
    print(state["draft"])
    print("\nOptions:")
    print("1 → approve")
    print("2 → edit")

    choice = input("Select option: ")

    if choice == "1":
        return {"final": state["draft"], "decision": "approve"}

    else:
        edited = llm.invoke("Please edit the following draft answer to improve it:\n\n" + state["draft"]).content
        return {"final": edited, "decision": "edit" }

In [12]:
def router(state: Tweetstate):
    return state["decision"]

builder = StateGraph(Tweetstate)

builder.add_node("draft_node", generate_draft)
builder.add_node("review_node", human_review)

builder.add_edge(START, "draft_node")
builder.add_edge("draft_node", "review_node")

builder.add_conditional_edges(
    "review_node",
    router,
    {
        "approve": END,
        "edit": END
    }
)

graph = builder.compile()

In [13]:
result = graph.invoke({
        "messages": [HumanMessage(content="Explain RAG in simple words")]
    })

print("\n=== FINAL OUTPUT ===")
print(result["final"])


--- HUMAN REVIEW ---
Draft answer:

Imagine you have a **super-smart student** (that's like an AI chatbot, or LLM – Large Language Model).

This student is brilliant and has read **tons of books** (its training data). It can answer almost any general question based on what it remembers.

**The Problem:**
*   If you ask the student about something **very new** (like yesterday's news) or **very specific** (like your family's secret recipe), it won't know because it wasn't in its original books.
*   It might even try to **guess** and make something up (this is called "hallucination" in AI).

**Here's where RAG comes in:**

**RAG stands for "Retrieval Augmented Generation."**

Think of it like giving our super-smart student a **personal research assistant and a library card** *right before* they answer a question:

1.  **Retrieval (R):** When you ask a question, the student's research assistant first **goes to a specific library** (which could be your company documents, a live website, a 